# 📝 질의 변환 과제 LV2(응용)

교안 02의 질의 변환 다섯 가지를 차례로 적용합니다. 한글로 작성한 가상 생활용품 쇼핑몰 도움말 12개(`lv2_docs.json`)를 사용합니다. 배송·교환반품·주문결제·회원 안내이며, 본문과 운영 규칙은 모두 수업용 창작입니다. LV1에서 만든 하이브리드 검색기는 준비되어 있고, 이 과제에서는 질의 변환 검색기와 체인을 직접 구성합니다.

- 1~2번: Self-Query로 검색어와 조건을 나누고, 같은 조건을 BM25·Dense에 함께 적용
- 3~6번: HyDE·Multi-Query·Step-back·Decomposition의 검색기·체인 구성과 검색 입력 연결
- 7번: 원질문과 실제 검색 원문을 연결해 답변 생성(기록 저장 제공)
- 8번: 질문의 특성에 맞는 질의 변환 선택(서술)

`.env`의 OpenAI 키가 필요합니다. GPT 요청은 모두 6회(1·3·4·5·6·7번에 한 번씩)이고, 임베딩은 문서 12개와 검색 질문마다 요청합니다. 질문·검색 결과 조립·Recall 계산·출력·저장은 제공 코드입니다. 변환 후 결과가 항상 좋아지는 것은 아니므로 출력된 질문과 원문을 읽어 보세요.

**풀이 방법**: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 검색기 생성과 핵심 처리 코드를 작성합니다. `...`는 호출식 전체나 여러 줄의 코드로 바꿀 수 있습니다. 입력 준비·결과 조립·비교·출력 코드는 완성되어 있습니다. 문항별 핵심 동작만 자가채점하며, 검색 순위·Recall 수치·모델의 문장 표현은 고정하지 않습니다. 앞 문항의 검색기와 결과를 다음 문항에서 이어 씁니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 이 셀은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("lv2_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())


아래 준비 셀을 순서대로 실행하세요. 교안 02와 같은 질의 변환 도구, 벡터 저장소, 하이브리드 검색기, Self-Query 추출 규칙 `query_schema_prompt`(교안 02 따라하기와 같은 규칙), 전후 비교 함수 `compare_recall`, 중복 제거 함수 `unique_documents`를 준비합니다.


In [ ]:
# 질의 생성과 메타데이터 조건 번역에 쓰는 도구를 가져옵니다.
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from pydantic import BaseModel, Field


In [ ]:
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 -> 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lv2_shop", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lv2_shop 적재 수:", len(added_ids))


In [ ]:
# LV1에서 연습한 하이브리드 검색기입니다. 문서 12개 중 BM25·Dense는 각각 최대 TOP_K=2개를 가져옵니다.
TOP_K = 2
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=TOP_K)
dense = vector_store.as_retriever(search_kwargs={"k": TOP_K})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")


In [ ]:
# 검색어와 조건을 추출하는 규칙입니다.
query_schema_prompt = PromptTemplate.from_template(
    "마지막 User Query를 검색어와 메타데이터 조건으로 나누세요. 예시의 값을 복사하지 마세요. "
    "JSON 객체 하나만 출력하세요. query에는 본문에서 찾을 핵심 주제를, filter에는 명시된 조건을 넣습니다. "
    "주제가 있으면 query를 비우지 말고, 조건이 없을 때만 filter를 NO_FILTER로 적으세요. "
    "filter는 비교 연산자({allowed_comparators})와 논리 연산자({allowed_operators})로 표현합니다. "
    "Data Source에 정의된 필드만 사용하며, 숫자는 문자열로 바꾸지 마세요."
)


In [ ]:
# 같은 질문에서 변환 전후에 찾은 정답 원문의 비율을 비교합니다.
def compare_recall(before, after, expected_ids):
    """변환 전후의 전체 후보 Recall과 중복을 제외한 문서 수를 비교합니다."""
    expected_ids = set(expected_ids)
    rows = []
    for stage, results in [("변환 전", before), ("변환 후", after)]:
        retrieved_ids = {doc.metadata["source_id"] for doc in results}
        rows.append({
            "stage": stage,
            "recall": len(retrieved_ids & expected_ids) / len(expected_ids),
            "retrieved_count": len(retrieved_ids),
            "expected_count": len(expected_ids),
            "missing_ids": sorted(expected_ids - retrieved_ids),
        })
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head().round(3))


In [ ]:
# 여러 질문이 같은 원문을 찾아도 답변 문맥에는 한 번만 넣습니다.
def unique_documents(documents):
    """원문 ID를 기준으로 첫 등장 순서를 유지하며 중복을 제거합니다."""
    by_id = {}
    for doc in documents:
        source_id = doc.metadata["source_id"]
        if source_id not in by_id:
            by_id[source_id] = doc
    return list(by_id.values())


## 1. 메타데이터 조건을 추출하는 Self-Query 검색기를 만듭니다

**배경**: 찾을 내용과 구매 경로·안내 분야 조건을 나눠 온라인 배송 안내에서만 검색하려고 합니다.

**요구사항**:

- **`self_query`**: 모델·벡터 저장소·메타데이터 설명을 연결한 `SelfQueryRetriever`를 만드세요. 필요한 설정값은 작성 구간의 주석에 있습니다.

**확인 기준**: `self_query`는 준비된 `vector_store`에 연결된 `SelfQueryRetriever` 객체입니다. 아래 코드가 조건 추출과 검색을 실행합니다.

<details><summary>힌트</summary>

```text
접근방법:
- SelfQueryRetriever.from_llm에 검색에 사용할 모델·저장소와 조건으로 쓸 필드를 연결합니다.

세부구현:
1. llm, vectorstore, metadata_field_info에 준비된 값을 연결합니다.
2. 문서 설명, 추출 규칙, 조건 번역기와 검색 개수는 코드 주석을 참고합니다.
```

</details>


In [ ]:
# 1) 조건으로 사용할 메타데이터 필드와 값의 의미를 준비합니다.
metadata_field_info = [
    AttributeInfo(name="category", description="안내 분야: 배송, 교환반품, 주문결제, 회원", type="string"),
    AttributeInfo(name="channel", description="구매 경로: 온라인, 매장", type="string"),
]
question = "온라인 구매의 배송 분야 안내에서 합배송 조건과 배송 상태 확인 방법을 찾아 주세요."

# ====================================================================
# [작성] 2) SelfQueryRetriever.from_llm으로 검색기를 만드세요.
# llm: 준비된 llm / vectorstore: 준비된 vector_store
# document_contents: "생활용품 쇼핑몰의 배송·교환·반품·결제 안내" / metadata_field_info: 위 필드 설명 목록
# chain_kwargs: {"schema_prompt": query_schema_prompt} (조건 추출 규칙)
# structured_query_translator: ChromaTranslator() (조건을 Chroma 필터로 번역)
# search_kwargs: {"k": TOP_K} (검색 개수)
self_query = ...
# ====================================================================

# 3) 조건을 한 번 추출하고 2번 문제에서도 같은 결과를 재사용합니다.
# parsed는 문서 목록이 아니라 query·filter 속성을 가진 구조화 객체입니다.
parsed = self_query.query_constructor.invoke({"query": question})
query_text, search_kwargs = ChromaTranslator().visit_structured_query(parsed)
where = search_kwargs.get("filter")
print("검색어:", query_text)
print("조건:", where)

# 4) 본문 검색어와 필터를 함께 전달해 조건에 맞는 원문을 찾습니다.
self_query_results = vector_store.similarity_search(query_text, k=TOP_K, filter=where)
show_results(self_query_results)


In [ ]:
# [자가채점]
assert isinstance(self_query, SelfQueryRetriever), "SelfQueryRetriever.from_llm으로 검색기를 만드세요."
assert self_query.vectorstore is vector_store, "vectorstore에 준비된 vector_store를 연결하세요."
print("✅ 1번 핵심 확인 완료!")


## 2. 두 검색에 같은 메타데이터 조건을 적용합니다

**배경**: 하이브리드 검색 결과 전체가 조건을 지키려면 BM25와 Dense 양쪽에 같은 조건을 적용해야 합니다.

**요구사항**:

- **`scored_candidates`와 `filtered_dense`**: 1번의 조건을 두 검색에 똑같이 적용하세요. BM25는 조건에 맞는 문서·점수만 남기고, Dense는 같은 `where` 필터로 검색하세요.

**확인 기준**: 두 결과에 문서가 하나 이상 있으며, 문서 ID가 모두 `allowed_ids`에 속합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 조건을 BM25 후보와 Dense 검색에 각각 적용합니다.

세부구현:
1. zip으로 문서와 점수를 묶고 source_id가 allowed_ids에 있는 항목만 남깁니다.
2. similarity_search에 query_text, k, filter를 전달합니다.
```

</details>


In [ ]:
# 1) 1번의 where 조건을 만족하는 원문 ID를 조회합니다.
# allowed_ids는 두 검색에서 공통으로 사용할 허용 문서 ID 집합입니다.
matched = vector_store.get(where=where, include=["metadatas"])
allowed_ids = {item["source_id"] for item in matched["metadatas"]}
# 기존 BM25의 전체 문서 점수입니다. bm25.docs와 scores의 순서가 같습니다.
scores = bm25.vectorizer.get_scores(bm25.preprocess_func(query_text))

# ====================================================================
# [작성] 2) 두 검색에 같은 조건을 적용하세요.
# scored_candidates: (Document, 점수) 쌍의 목록입니다.
# zip(bm25.docs, scores)를 순회해 doc.metadata["source_id"]가 allowed_ids에 있는 쌍만 남깁니다.
# filtered_dense: vector_store.similarity_search의 Document 목록입니다.
# 검색어는 query_text, k는 TOP_K, filter는 where를 사용합니다.
scored_candidates = ...
filtered_dense = ...
# ====================================================================

# 3) BM25 후보를 점수순으로 정렬한 뒤 같은 조건의 Dense 결과와 합칩니다.
scored_candidates.sort(key=lambda item: item[1], reverse=True)
filtered_bm25 = [doc for doc, score in scored_candidates[:TOP_K]]
filtered_results = hybrid.weighted_reciprocal_rank([filtered_bm25, filtered_dense])[:TOP_K]

# 4) 조건 적용 전후의 원문과 Recall을 출력합니다.
expected_ids = {"shop03", "shop12"}
before_results = hybrid.invoke(question)[:TOP_K]
print("조건 없이 검색")
show_results(before_results)
print("같은 조건으로 검색")
show_results(filtered_results)
compare_recall(before_results, filtered_results, expected_ids)


In [ ]:
# [자가채점]
assert scored_candidates and all(doc.metadata["source_id"] in allowed_ids for doc, score in scored_candidates), "BM25 후보에는 allowed_ids에 속한 문서만 남기세요."
assert filtered_dense and all(doc.metadata["source_id"] in allowed_ids for doc in filtered_dense), "Dense 검색에도 where 필터를 적용하세요."
print("✅ 2번 핵심 확인 완료!")


아래는 교안 02의 HyDE 프롬프트 `hyde_prompt`입니다. 가상문단을 한글로 생성하는 규칙을 준비하며, 체인은 3번에서 직접 구성합니다.


In [ ]:
# 가상문서는 검색에만 쓰고 최종 답변은 실제 원문으로 작성합니다.
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "질문에 답할 내용이 담긴 가상의 원문을 3문장 이내로 작성하세요. "
               "각 요구에 필요한 핵심 개념을 해당 분야의 표준 용어로 명시하고 설명하세요. "
               "질문을 다시 쓰거나 관련 주제를 추가하지 마세요. "
               "도서 검색 질문에는 해당 내용을 가르치는 책의 소개문을 쓰되 책 제목·저자·출판사는 만들지 마세요. "
               "확인되지 않은 구체적인 수치나 규정을 지어내지 마세요. "
               "요청문·검색어 목록 없이 한글 원문만 출력하세요."),
    ("human", "{question}"),
])


## 3. HyDE의 문단 생성과 Dense 검색을 연결합니다

**배경**: 환불이라는 용어를 쓰지 않은 질문을 설명 문단으로 바꿔 관련 안내를 검색합니다.

**요구사항**:

- **`hyde_chain`과 `hyde_dense`**: 가상문단 생성 체인을 만들고, 생성된 `hypothesis`로 Dense 검색을 실행하세요.

**확인 기준**: `hyde_dense`에는 가상문단으로 검색한 실제 `Document`가 하나 이상 담깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 프롬프트·모델·문자열 파서를 연결하고, 생성 결과를 검색 입력으로 사용합니다.

세부구현:
1. hyde_prompt, llm, StrOutputParser를 순서대로 연결합니다.
2. 생성된 hypothesis를 dense의 invoke에 전달합니다.
```

</details>


In [ ]:
# 비교할 질문과 정답 원문 ID입니다.
hyde_question = "물건을 돌려보낸 뒤 돈은 어떤 과정을 거쳐 돌아오나요?"
hyde_expected_ids = {"shop06", "shop07"}

# ====================================================================
# [작성] 1) 가상문단 생성 체인을 구성하세요.
# hyde_prompt -> llm -> StrOutputParser() 순서로 | 연산자를 사용해 연결합니다.
hyde_chain = ...
# ====================================================================

# 2) 질문에 관한 한글 가상문단을 생성합니다.
# hypothesis는 검색을 돕는 가상문단이며 실제 원문이 아닙니다.
hypothesis = hyde_chain.invoke({"question": hyde_question})
print("검색용 가상문단:", hypothesis)

# BM25는 모델이 추가한 표현이 아니라 원질문의 용어로 검색합니다.
hyde_bm25 = bm25.invoke(hyde_question)

# ====================================================================
# [작성] 3) dense.invoke를 호출해 가상문단 hypothesis로 검색하세요.
# 원질문은 BM25에서 검색했고, Dense에는 생성된 설명 문단을 전달합니다.
# 반환값은 실제 원문 Document 목록입니다.
hyde_dense = ...
# ====================================================================

# 원질문으로 검색한 Dense 결과도 비교용으로 받습니다.
question_dense = dense.invoke(hyde_question)
print("Dense(가상문단):", [doc.metadata["source_id"] for doc in hyde_dense])
print("Dense(원질문):", [doc.metadata["source_id"] for doc in question_dense])

# BM25 결과, 가상문단 Dense 결과 순서로 RRF에 넣습니다.
hyde_results = hybrid.weighted_reciprocal_rank([hyde_bm25, hyde_dense])[:TOP_K]
show_results(hyde_results)
hyde_before = hybrid.invoke(hyde_question)[:TOP_K]
compare_recall(hyde_before, hyde_results, hyde_expected_ids)


In [ ]:
# [자가채점]
assert isinstance(hyde_dense, list) and hyde_dense, "가상문단 hypothesis를 dense로 검색한 원문 목록을 hyde_dense에 담으세요."
print("✅ 3번 핵심 확인 완료!")


아래는 교안 02의 Multi-Query 프롬프트 `multi_prompt`와, 생성 질문을 출력하는 `show_queries`입니다.


In [ ]:
# from_llm의 기본 줄 단위 파서가 읽을 수 있게 번호 없이 한 줄에 하나씩 받습니다.
multi_prompt = ChatPromptTemplate.from_template(
    "질문과 같은 의미를 유지하는 검색 질문을 서로 다른 표현으로 정확히 3개 작성하세요. "
    "일상 표현을 관련 분야의 표준 용어로 바꾼 질문도 포함하세요. "
    "원문의 정보 요구·고유명사·조건을 유지하세요. 질문의 언어를 유지하세요. "
    "설명·번호·빈 줄 없이 한 줄에 질문 하나만 출력하세요.\n질문: {question}"
)


In [ ]:
def show_queries(queries):
    """실제로 검색에 사용할 생성 질문을 출력하고 그대로 전달합니다."""
    # 출력용으로 LLM을 다시 호출하지 않고 기존 체인의 결과를 관찰합니다.
    print("생성 질문:", queries)
    return queries


## 4. 원질문도 함께 검색하는 Multi-Query 검색기를 만듭니다

**배경**: 같은 뜻을 여러 표현으로 검색하면 원질문의 표현만으로 놓친 문서를 찾을 수 있습니다.

**요구사항**:

- **`multi_query`**: `MultiQueryRetriever.from_llm`으로 질문을 여러 표현으로 바꿔 검색하는 검색기를 만드세요. 원질문도 검색하도록 설정하세요.

**확인 기준**: `multi_query`는 `hybrid`에 연결된 `MultiQueryRetriever`이며, `include_original`이 `True`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 생성에 쓸 모델·프롬프트와 실제 검색기를 연결합니다.

세부구현:
1. retriever에는 hybrid, llm에는 llm, prompt에는 multi_prompt를 연결합니다.
2. include_original로 원질문 검색을 포함합니다.
```

</details>


In [ ]:
# 1) 같은 뜻의 다른 표현으로 검색할 한글 질문입니다.
multi_question = "여러 물건을 한 상자로 묶어 받는 조건이 궁금해요."
multi_expected_ids = {"shop03"}

# ====================================================================
# [작성] 2) MultiQueryRetriever.from_llm으로 검색기를 만드세요.
# retriever에는 hybrid, llm에는 llm, prompt에는 multi_prompt를 연결합니다.
# include_original=True로 원질문도 검색에 포함합니다.
multi_query = ...
# ====================================================================

# 3) 생성된 질문을 출력하고 같은 원문을 한 번만 남기는 처리를 연결합니다.
multi_query.llm_chain = multi_query.llm_chain | show_queries
multi_query_chain = multi_query | unique_documents
multi_results = multi_query_chain.invoke(multi_question)

# 4) 여러 질문의 전체 결과를 합친 목록과 원질문 결과를 비교합니다.
multi_before = hybrid.invoke(multi_question)
show_results(multi_results)
compare_recall(multi_before, multi_results, multi_expected_ids)


In [ ]:
# [자가채점]
assert isinstance(multi_query, MultiQueryRetriever), "MultiQueryRetriever.from_llm으로 검색기를 만드세요."
assert multi_query.retriever is hybrid and multi_query.include_original, "hybrid를 연결하고 include_original=True를 설정하세요."
print("✅ 4번 핵심 확인 완료!")


아래 Step-back 프롬프트는 쇼핑몰 업무 범위를 유지하면서 구체적인 사례를 배경 개념에 관한 질문으로 바꿉니다. 체인은 5번에서 구성합니다.


In [ ]:
# 쇼핑몰 업무 범위를 유지하며 구체적인 사례를 배경 개념으로 바꿉니다.
stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 특정 상품이나 주문 상황을 일반화해, 그 배경 개념을 묻는 질문 하나를 작성하세요. "
               "구매·배송·교환·반품이라는 업무 범위를 유지하고, 질문의 핵심 처리와 그 목적을 남기세요. "
               "관련 없는 법률·경제 이론으로 주제를 넓히지 마세요. 답변이나 설명 없이 한글 질문만 출력하세요."),
    ("human", "택배가 어디까지 왔는지 알려면 왜 운송장번호가 필요한가요?"),
    ("ai", "운송장 식별자와 배송 이력 추적은 어떤 관계인가요?"),
    ("human", "{question}"),
])


## 5. Step-back 체인을 만들고 두 질문을 검색합니다

**배경**: 반품 처리 절차와 검수의 목적을 함께 찾기 위해 구체적인 질문과 배경 질문을 각각 검색합니다.

**요구사항**:

- **`stepback_chain`과 `question_results`**: 배경 질문을 만드는 체인을 구성하고, 원질문과 배경 질문을 각각 검색하세요.

**확인 기준**: `question_results`에 문서 목록 2개가 담깁니다. 원질문 결과, 배경 질문 결과 순서입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자열을 생성하는 체인과 여러 입력을 검색하는 batch를 연결합니다.

세부구현:
1. stepback_prompt, llm, StrOutputParser로 체인을 구성합니다.
2. 원질문과 배경 질문의 목록을 hybrid.batch에 전달합니다.
```

</details>


In [ ]:
# 구체적인 환불 절차와 검수의 목적을 함께 묻는 질문입니다.
stepback_question = "반품한 상품을 확인한 다음에 환불하는 이유는 무엇이며, 어떤 절차를 거치나요?"
stepback_expected_ids = {"shop06", "shop07"}

# ====================================================================
# [작성] 1) 배경 질문 생성 체인을 구성하세요.
# stepback_prompt -> llm -> StrOutputParser() 순서로 연결합니다.
stepback_chain = ...
# ====================================================================

# 2) 원질문의 배경 개념을 묻는 한글 질문을 생성합니다.
# background_question에는 구체적인 사례의 배경 개념을 묻는 질문이 담깁니다.
background_question = stepback_chain.invoke({"question": stepback_question})
print("배경 질문:", background_question)

# ====================================================================
# [작성] 3) hybrid.batch를 호출해 원질문과 배경 질문을 함께 검색하세요.
# 입력 목록은 [stepback_question, background_question] 순서입니다.
# 반환값도 같은 순서로 검색 결과 목록 두 개를 담습니다.
question_results = ...
# ====================================================================


# 첫 번째는 원질문, 두 번째는 배경 질문의 결과입니다.
direct_results = question_results[0][:TOP_K]
background_results = question_results[1][:TOP_K]

# 배경 질문이 새로 찾은 원문만 보도록 ID 집합의 차를 구합니다.
direct_ids = {doc.metadata["source_id"] for doc in direct_results}
background_ids = {doc.metadata["source_id"] for doc in background_results}
added_ids = background_ids - direct_ids
print("배경 질문으로 추가된 원문 ID:", added_ids)
# 원질문 결과를 먼저 놓고 같은 원문은 한 번만 남깁니다.
stepback_results = unique_documents(direct_results + background_results)
show_results(stepback_results)
compare_recall(direct_results, stepback_results, stepback_expected_ids)


In [ ]:
# [자가채점]
assert isinstance(question_results, list) and len(question_results) == 2, "hybrid.batch에 질문 두 개를 전달해 검색 결과 두 묶음을 받으세요."
assert all(isinstance(results, list) for results in question_results), "질문별 Document 목록 두 개가 담겨야 합니다."
print("✅ 5번 핵심 확인 완료!")


아래는 질문 분해 프롬프트 `decompose_prompt`와 출력 형식 `SubQuestions`입니다. `questions`에 하위 질문 2~3개를 담도록 정의하며, 체인은 6번에서 구성합니다.


In [ ]:
class SubQuestions(BaseModel):
    """원질문을 나누어 검색할 하위 질문 목록입니다."""
    questions: list[str] = Field(
        description=(
            "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개. "
            "각 질문만으로 검색할 수 있게 대상과 조건을 포함하고, "
            "원질문에 없는 요구를 추가하지 않으며 같은 언어로 작성."
        ),
        min_length=2, max_length=3,
    )


# 스키마는 목록 모양을 제어합니다. 질문이 원의도를 보존했는지는 사람이 읽습니다.
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개로 나누세요. "
               "각 질문만으로 검색할 수 있게 대상과 조건을 포함하세요. 원질문에 없는 요구를 추가하지 말고 같은 언어로 작성하세요."),
    ("human", "{question}"),
])


## 6. 질문 분해 체인을 만들고 하위 질문을 검색합니다

**배경**: 교환 신청과 배송지 변경을 함께 묻는 질문을 나눠 각 요구의 근거를 찾습니다.

**요구사항**:

- **`decompose_chain`과 `subquestion_results`**: `SubQuestions` 형식으로 질문을 분해하는 체인을 구성하고, 하위 질문들을 각각 검색하세요.

**확인 기준**: `subquestion_results`에 하위 질문마다 문서 목록 하나가 담깁니다. 바깥 목록 길이는 `subquestions.questions`와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 구조화 출력을 사용해 하위 질문 목록을 받은 뒤 batch로 검색합니다.

세부구현:
1. llm.with_structured_output에 SubQuestions와 json_schema 방식을 지정합니다.
2. decompose_prompt와 연결한 체인 결과의 questions를 검색에 사용합니다.
```

</details>


In [ ]:
# 교환 신청과 배송지 변경이라는 서로 다른 두 요구를 묻는 질문입니다.
decomposition_question = "깨진 컵을 교환하려면 어떤 자료를 제출해야 하나요? 배송지를 바꿀 수 있는 시점도 알려 주세요."
decomposition_expected_ids = {"shop04", "shop02"}

# ====================================================================
# [작성] 1) 하위 질문 목록을 구조화 출력으로 받는 체인을 만드세요.
# decompose_prompt 뒤에 llm.with_structured_output을 연결합니다.
# 출력 스키마는 SubQuestions, method는 "json_schema"를 사용합니다.
decompose_chain = ...
# ====================================================================

# 2) 서로 다른 요구를 한글 하위 질문으로 나눕니다.
# 반환 객체의 questions 속성에 검색할 하위 질문 목록이 담깁니다.
subquestions = decompose_chain.invoke({"question": decomposition_question})
print("하위 질문:", subquestions.questions)


# ====================================================================
# [작성] 3) hybrid.batch에 subquestions.questions를 전달합니다.
# 결과는 하위 질문마다 Document 목록 하나를 담은 중첩 목록입니다.
subquestion_results = ...
# ====================================================================


# 질문별 앞 TOP_K개를 하나의 목록으로 모읍니다.
all_results = []
for results in subquestion_results:
    all_results.extend(results[:TOP_K])
# 여러 질문이 찾은 같은 원문은 한 번만 남깁니다.
decomposition_results = unique_documents(all_results)
show_results(decomposition_results)
# 원질문 한 번의 검색과 분해 후 결과를 비교합니다.
decomposition_before = hybrid.invoke(decomposition_question)[:TOP_K]
compare_recall(decomposition_before, decomposition_results, decomposition_expected_ids)


In [ ]:
# [자가채점]
assert isinstance(subquestion_results, list) and len(subquestion_results) == len(subquestions.questions), 'hybrid.batch에 하위 질문 목록을 전달해 질문마다 결과를 받으세요.'
assert all(isinstance(results, list) for results in subquestion_results), '질문별 검색 결과 목록을 담은 중첩 목록이어야 합니다.'
print("✅ 6번 핵심 확인 완료!")


아래는 교안 02의 답변 문맥 함수 `format_context`와 답변 체인 `answer_chain`입니다. 답변은 `[원문 ID]`를 인용합니다.


In [ ]:
# 본문과 조건 메타데이터를 함께 줘 안내 분야·구매 경로도 답변에서 확인할 수 있게 합니다.
def format_context(documents):
    """실제 검색 문서의 본문·메타데이터·출처를 답변 문맥으로 만듭니다."""
    return "\n\n".join(
        f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
        f"메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for doc in documents
    )


In [ ]:
# 실제 검색 원문에 있는 내용으로만 원질문에 답합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 본문과 메타데이터에서 직접 확인되는 내용만 한글로 답하세요. "
               "질문에 직접 답하는 내용만 최대 3개 항목으로 쓰고, 항목마다 1~2문장과 [원문 ID] 인용을 넣으세요. 관련 없는 문서는 언급하지 마세요. "
               "원문에 없는 절차·조건·조언을 추가하지 마세요. "
               "원문의 권고나 가능성을 의무로 바꾸지 말고, 수치·단위·비율·조건을 그대로 보존하세요. 사용자가 제시한 방안을 원문이 허용하거나 정당화한다고 추론하지 마세요. "
               "근거가 부족한 부분은 확인할 수 없다고 말하고, 추가 결론은 쓰지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


## 7. 실제 검색 원문을 답변 근거로 연결합니다

**배경**: 답변 모델이 실제 검색한 문서를 읽도록 원문 목록을 하나의 근거 문자열로 만듭니다.

**요구사항**:

- **`answer`**: 준비된 `answer_chain`을 호출해 원질문 `decomposition_question`에 답하세요. `context`에는 실제 검색 원문 `decomposition_results`를 `format_context`로 변환해 전달하세요.

**확인 기준**: `answer`는 비어 있지 않은 한글 답변이며, 검색한 원문 ID를 `[shop04]`처럼 하나 이상 인용합니다. 문장 표현은 달라도 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 답변 체인의 입력 딕셔너리에 원질문과 실제 원문을 연결합니다.

세부구현:
1. question에는 decomposition_question을 사용합니다.
2. context에는 format_context로 만든 실제 원문 문자열을 사용합니다.
```

</details>


In [ ]:
# 1) 6번의 원질문과 검색 원문을 그대로 사용합니다.
# decomposition_question: 교환 신청과 배송지 변경을 함께 묻는 원질문 문자열
# decomposition_results: 하위 질문들로 찾은 실제 Document 목록
# format_context: Document 목록을 원문 ID·메타데이터·본문이 있는 문자열로 바꾸는 함수

# ====================================================================
# [작성] 2) 원질문과 실제 검색 원문을 연결해 답변을 생성하세요.
# answer_chain.invoke는 question과 context 키가 있는 딕셔너리를 받습니다.
# question에는 원질문, context에는 format_context로 변환한 검색 원문을 전달합니다.
answer = ...
# ====================================================================

# 3) 답변을 출력하고 검색 과정과 함께 저장합니다.
print(answer)
# source_ids는 답변에 인용됐는지와 관계없이 검색된 원문 전체의 ID입니다.
report = {
    "question": decomposition_question,
    "subquestions": subquestions.questions,
    "source_ids": [doc.metadata["source_id"] for doc in decomposition_results],
    "answer": answer,
}
save_json("shop_search_report.json", report)


In [ ]:
# [자가채점]
assert isinstance(answer, str) and answer.strip(), "answer_chain에 원질문과 실제 원문을 전달해 답변 문자열을 받으세요."
assert any(f"[{doc.metadata['source_id']}]" in answer for doc in decomposition_results), "실제 검색한 원문 ID를 인용하는지 확인하세요."
print("✅ 7번 핵심 확인 완료!")


## 8. 조건이 있는 질문에 맞는 질의 변환을 고릅니다

**배경**: 매장 구매와 온라인 구매의 안내가 함께 있어 질문에 맞는 구매 경로로 검색 범위를 좁혀야 합니다.

**요구사항**:

- **방법 선택**: “매장에서 구매한 상품의 교환·반품 안내”를 찾는 데 적절한 질의 변환 하나를 고르고 이유를 한 문장으로 쓰세요.

**확인 기준**: 구매 경로와 안내 분야를 검색 조건에 반영하는 이유를 설명하면 됩니다. 코드는 작성하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 본문에서 찾을 주제와 메타데이터로 제한할 조건을 구분합니다.

세부구현:
1. 질문에서 구매 경로와 안내 분야를 찾습니다.
2. 그 조건을 검색에 전달할 수 있는 방법을 고릅니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
